# 🛰️🌾 AgriOrbit — Neural Rain Model (Colab training)

Trains the LSTM rain-forecast model used by the AgriOrbit backend on **ERA5 satellite reanalysis**
(via the free Open-Meteo archive) covering **1991–2024** for **28 Indian agro-climatic stations**.

**Outputs** (downloaded at the end — place all three in `backend/ml/artifacts/`):
- `rain_model.onnx` — the trained model (PyTorch-free inference via onnxruntime)
- `scaler.json` — feature normalisation statistics
- `metrics.json` — validation metrics the web app shows on its model card

**How to run**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`  (≈15–25 min: data download dominates)
3. Your browser downloads `agriorbit_artifacts.zip` at the end.

> **Responsible-AI note.** This model is a statistical companion to the official numerical
> forecast, not a replacement. Validation metrics (vs a climatology baseline) are published
> in-app so users can judge its reliability themselves.

## 0 · Setup

In [ ]:
%pip install -q onnx onnxruntime pandas pyarrow scikit-learn matplotlib tqdm requests

import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
if device != 'cuda':
    print('⚠️  No GPU found → Runtime → Change runtime type → T4 GPU')

## 1 · Configuration

The feature list is the single source of truth shared with the backend
(`backend/app/services/openmeteo.py::FEATURE_NAMES`). Every variable is a **daily** aggregate
available under the identical name in both the archive API (training) and the forecast API
(inference), so train/serve skew is impossible by construction.

In [ ]:
STATIONS = {  # 28 stations across India's agro-climatic zones
    'Delhi': (28.61, 77.21), 'Amritsar': (31.63, 74.87), 'Ludhiana': (30.90, 75.85),
    'Jaipur': (26.91, 75.79), 'Lucknow': (26.85, 80.95), 'Kanpur': (26.45, 80.33),
    'Varanasi': (25.32, 82.99), 'Patna': (25.59, 85.14), 'Bhopal': (23.26, 77.41),
    'Indore': (22.72, 75.86), 'Ahmedabad': (23.03, 72.58), 'Surat': (21.17, 72.83),
    'Mumbai': (19.08, 72.88), 'Pune': (18.52, 73.86), 'Nagpur': (21.15, 79.09),
    'Hyderabad': (17.38, 78.49), 'Vijayawada': (16.51, 80.65), 'Chennai': (13.08, 80.27),
    'Coimbatore': (11.02, 76.96), 'Madurai': (9.93, 78.12), 'Bengaluru': (12.97, 77.59),
    'Thiruvananthapuram': (8.52, 76.94), 'Kolkata': (22.57, 88.36), 'Bhubaneswar': (20.30, 85.82),
    'Ranchi': (23.36, 85.33), 'Guwahati': (26.14, 91.74), 'Raipur': (21.25, 81.63),
    'Dehradun': (30.32, 78.03),
}

START, END        = '1991-01-01', '2024-12-31'
VAL_START_YEAR    = 2019                      # 2019-2024 held out for validation
LOOKBACK, HORIZON = 30, 7
RAIN_THRESHOLD_MM = 1.0                       # 'rain day' definition
FEATURES = [  # (field name, Open-Meteo daily variable) — ORDER MATTERS
    ('tmax',      'temperature_2m_max'),
    ('tmin',      'temperature_2m_min'),
    ('precip',    'precipitation_sum'),
    ('et0',       'et0_fao_evapotranspiration'),
    ('radiation', 'shortwave_radiation_sum'),
    ('wind',      'wind_speed_10m_max'),
    ('rh',        'relative_humidity_2m_mean'),
    ('pressure',  'surface_pressure_mean'),
    ('sm0_7',     'soil_moisture_0_to_7cm_mean'),
    ('sm7_28',    'soil_moisture_7_to_28cm_mean'),
]
BATCH, EPOCHS, LR, PATIENCE = 512, 10, 1e-3, 3

DATA_DIR = Path('/content/agriorbit_data');     DATA_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path('/content/agriorbit_artifacts'); OUT_DIR.mkdir(exist_ok=True)
print(f'{len(STATIONS)} stations · {START}[:4]–{END[:4]} · features={len(FEATURES)}')

## 2 · Download the ERA5 archive (one request per station)

Each station is one API call (~12 400 daily rows), cached to parquet so re-runs are instant.

In [ ]:
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'

def download_station(name: str, lat: float, lon: float) -> pd.DataFrame:
    cache = DATA_DIR / f"{name.lower().replace(' ', '_')}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': START, 'end_date': END,
        'daily': ','.join(api for _, api in FEATURES),
        'timezone': 'GMT',
    }
    for attempt in range(5):
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=600)
            r.raise_for_status()
            break
        except Exception as e:
            if attempt == 4: raise
            wait = 10 * (attempt + 1)
            print(f'  retry {name} in {wait}s ({e})')
            time.sleep(wait)
    daily = r.json()['daily']
    df = pd.DataFrame(daily)
    df['station'] = name
    df.to_parquet(cache)
    return df

frames = []
for name, (lat, lon) in STATIONS.items():
    df = download_station(name, lat, lon)
    print(f'{name:22s} {len(df):6d} days   {df.time.iloc[0]} → {df.time.iloc[-1]}')
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data['time'] = pd.to_datetime(data['time'])
data = data.sort_values(['station', 'time']).reset_index(drop=True)
print('\nTotal rows:', len(data))

## 3 · Build supervised windows

Every sample = **30 input days → next 7 days**:
- classification target: `rain_day = precip ≥ 1 mm` per horizon day
- regression target: `log1p(total 7-day mm)` (log keeps monsoon extremes from dominating)

Split is by time, not random: **train 1991–2018, validation 2019–2024** — the model must prove
itself on years it has never seen.

In [ ]:
RAW = [api for _, api in FEATURES]
PRECIP_IDX = RAW.index('precipitation_sum')

def windows_for_station(df: pd.DataFrame):
    vals = df.sort_values('time')[RAW].to_numpy(np.float32)
    dates = df.sort_values('time')['time'].to_numpy()
    X, Yp, Yr, meta = [], [], [], []
    for t in range(LOOKBACK, len(vals) - HORIZON):
        win, fut = vals[t - LOOKBACK:t], vals[t:t + HORIZON, PRECIP_IDX]
        if np.isnan(win).any() or np.isnan(fut).any():
            continue
        X.append(win)
        Yp.append((fut >= RAIN_THRESHOLD_MM).astype(np.float32))
        Yr.append(np.float32(np.log1p(fut.sum())))
        meta.append(pd.Timestamp(dates[t]))          # meta date = day after window (first horizon day)
    return np.stack(X), np.stack(Yp), np.array(Yr, np.float32), meta

X_all, Yp_all, Yr_all, meta_all, months_all, stations_all = [], [], [], [], [], []
for name, df in data.groupby('station'):
    X, Yp, Yr, meta = windows_for_station(df)
    X_all.append(X); Yp_all.append(Yp); Yr_all.append(Yr)
    meta_all += meta
    months_all += [m.month for m in meta]
    stations_all += [name] * len(meta)

years = np.array([m.year for m in meta_all])
train_mask = years < VAL_START_YEAR

X_train = np.concatenate(X_all)[train_mask];  Yp_train = np.concatenate(Yp_all)[train_mask]
Yr_train = np.concatenate(Yr_all)[train_mask]
X_val  = np.concatenate(X_all)[~train_mask];  Yp_val = np.concatenate(Yp_all)[~train_mask]
Yr_val = np.concatenate(Yr_all)[~train_mask]
months_val = np.array(months_all)[~train_mask]; stations_val = np.array(stations_all)[~train_mask]

# normalisation from TRAIN only
mean = X_train.reshape(-1, len(FEATURES)).mean(axis=0)
std  = X_train.reshape(-1, len(FEATURES)).std(axis=0) + 1e-6
print(f'train windows: {len(X_train):,}   val windows: {len(X_val):,}')
print(f'radiation mean/std example: {mean[4]:.2f} / {std[4]:.2f}')

## 4 · Dataset & model

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, Yp, Yr):
        self.X  = torch.from_numpy(((X - mean) / std).astype(np.float32))
        self.Yp = torch.from_numpy(Yp.astype(np.float32))
        self.Yr = torch.from_numpy(Yr.astype(np.float32))
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Yp[i], self.Yr[i]

class RainLSTM(nn.Module):
    def __init__(self, n_features=len(FEATURES), hidden=128, layers=2, horizon=HORIZON, p=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers, batch_first=True, dropout=p)
        self.head_cls = nn.Linear(hidden, horizon)
        self.head_reg = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        h = out[:, -1]                      # last timestep summary of the 30-day window
        return self.head_cls(h), self.head_reg(h).squeeze(-1)

train_ds, val_ds = WindowDataset(X_train, Yp_train, Yr_train), WindowDataset(X_val, Yp_val, Yr_val)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=2048, shuffle=False)
model = RainLSTM().to(device)
print(model, f'\nparameters: {sum(p.numel() for p in model.parameters()):,}')

## 5 · Train (GPU, AMP, early stopping on validation Brier score)

Multi-task loss: weighted binary cross-entropy on rain-day logits (monsoon months are
class-imbalanced, hence `pos_weight`) + MSE on the log rainfall total.

In [ ]:
pos_rate = Yp_train.mean(axis=0)
pos_weight = torch.tensor((1 - pos_rate) / np.maximum(pos_rate, 1e-3), dtype=torch.float32, device=device)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

def brier(probs, targets): return float(np.mean((probs - targets) ** 2))

def evaluate():
    model.eval(); probs_all, t_all = [], []
    with torch.no_grad():
        for xb, yp, _ in val_dl:
            logits, _ = model(xb.to(device))
            probs_all.append(torch.sigmoid(logits).cpu().numpy()); t_all.append(yp.numpy())
    P, T = np.concatenate(probs_all), np.concatenate(t_all)
    return brier(P, T), float(((P >= 0.5) == (T > 0.5)).mean())

history, best, bad_epochs = {'loss': [], 'brier': [], 'acc': []}, float('inf'), 0
for epoch in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for xb, yp, yr in train_dl:
        xb, yp, yr = xb.to(device), yp.to(device), yr.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            logits, logtot = model(xb)
            loss = bce(logits, yp) + mse(logtot, yr)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item() * len(xb)
    sched.step()
    vb, va = evaluate()
    history['loss'].append(tot / len(train_ds)); history['brier'].append(vb); history['acc'].append(va)
    print(f'epoch {epoch:02d} | loss {history["loss"][-1]:.4f} | val Brier {vb:.4f} | val acc {va:.3f}')
    if vb < best - 1e-4:
        best, bad_epochs = vb, 0
        torch.save(model.state_dict(), OUT_DIR / 'best.pt')
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print('early stop'); break

model.load_state_dict(torch.load(OUT_DIR / 'best.pt', weights_only=True))
print('best val Brier:', round(best, 4))

## 6 · Honest evaluation — model vs climatology baseline

Baseline = the historical rain frequency of that station & month (what a farmer knows
“for free”). The model only earns its place if it beats this baseline on unseen years.
Curves and figures are saved into the artifact zip for the internship report.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

model.eval()
probs_val, logtot_val = [], []
with torch.no_grad():
    for xb, _, _ in val_dl:
        logits, logtot = model(xb.to(device))
        probs_val.append(torch.sigmoid(logits).cpu().numpy()); logtot_val.append(logtot.cpu().numpy())
P = np.concatenate(probs_val); T = Yp_val; LOGTOT = np.concatenate(logtot_val)

# climatology baseline per (station, month), from TRAIN windows only
key_rate = {}
for st, mo, yp in zip(np.array(stations_all)[train_mask], np.array(months_all)[train_mask], Yp_train):
    key_rate.setdefault((st, mo), []).append(yp)
key_rate = {k: np.stack(v).mean(axis=0) for k, v in key_rate.items()}
P_base = np.stack([key_rate[(st, mo)] for st, mo in zip(stations_val, months_val)])

metrics = {
    'val_years': '2019-2024',
    'val_windows': int(len(P)),
    'accuracy_at_50pct': round(float(((P >= 0.5) == (T > 0.5)).mean()), 4),
    'brier_score': round(brier(P, T), 4),
    'brier_baseline_climatology': round(brier(P_base, T), 4),
    'auc': round(float(roc_auc_score(T.ravel(), P.ravel())), 4),
    'per_horizon_brier': [round(brier(P[:, h], T[:, h]), 4) for h in range(HORIZON)],
}
print(json.dumps(metrics, indent=2))

# plots
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(history['loss'], label='train loss'); ax[0].set_title('training loss'); ax[0].legend()
ax[1].bar(range(1, HORIZON + 1), metrics['per_horizon_brier'])
ax[1].axhline(metrics['brier_baseline_climatology'], color='r', ls='--', label='climatology baseline')
ax[1].set_xlabel('forecast day'); ax[1].set_title('Brier per horizon (lower = better)'); ax[1].legend()
bins = np.linspace(0, 1, 11); pred_mean, true_rate, cnt = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (P[:, 0] >= lo) & (P[:, 0] < hi); cnt.append(m.sum())
    pred_mean.append(P[:, 0][m].mean() if m.any() else np.nan)
    true_rate.append(T[:, 0][m].mean() if m.any() else np.nan)
ax[2].plot([0, 1], [0, 1], 'k--'); ax[2].plot(pred_mean, true_rate, 'o-')
ax[2].set_title('reliability (day-1)'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
plt.tight_layout(); plt.savefig(OUT_DIR / 'evaluation.png', dpi=140); plt.show()

## 7 · Export to ONNX (+ scorer & metrics) and verify parity

The backend runs `onnxruntime` only — no PyTorch at inference. We verify the exported graph
reproduces PyTorch outputs to 1e-4 before packing.

In [ ]:
import onnxruntime as ort
from google.colab import files
import shutil

model.eval()
dummy = torch.zeros(1, LOOKBACK, len(FEATURES), device=device)
onnx_path = OUT_DIR / 'rain_model.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['features'],
    output_names=['rain_logits', 'log_total_mm'],
    dynamic_axes={'features': {0: 'batch'}, 'rain_logits': {0: 'batch'}, 'log_total_mm': {0: 'batch'}},
    opset_version=17,
)

# parity check PyTorch vs onnxruntime
sample = ((X_val[:64] - mean) / std).astype(np.float32)
with torch.no_grad():
    tl, tt = model(torch.from_numpy(sample).to(device))
ol, ot = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider']).run(None, {'features': sample})
print('max |Δlogits| =', float(np.abs(tl.cpu().numpy() - ol).max()))
print('max |Δtotal|  =', float(np.abs(tt.cpu().numpy() - ot).max()))

scaler = {
    'feature_names': [name for name, _ in FEATURES],
    'lookback': LOOKBACK, 'horizon': HORIZON,
    'rain_threshold_mm': RAIN_THRESHOLD_MM,
    'mean': [round(float(v), 6) for v in mean],
    'std':  [round(float(v), 6) for v in std],
    'target_transform': 'regression head predicts log1p(total_7d_mm) — apply expm1 at inference',
}
(OUT_DIR / 'scaler.json').write_text(json.dumps(scaler, indent=2))

metrics_doc = {
    'training': {
        'data': 'Open-Meteo ERA5/ERA5-Land archive (satellite-assimilated reanalysis)',
        'stations': len(STATIONS), 'period': f'{START[:4]}-{END[:4]}',
        'train_windows': int(len(X_train)), 'val_windows': int(len(X_val)),
        'split': f'train < {VAL_START_YEAR} / val ≥ {VAL_START_YEAR}',
        'framework': f'PyTorch {torch.__version__} (CUDA) → ONNX opset 17',
    },
    'metrics': metrics,
    'notes': 'Statistical companion to the numerical forecast; accuracy decays with lead time (see per-horizon Brier). Not a substitute for official IMD warnings.',
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics_doc, indent=2))

zip_path = shutil.make_archive('/content/agriorbit_artifacts', 'zip', OUT_DIR)
print('artifacts:', [p.name for p in OUT_DIR.iterdir()])
files.download(zip_path)

## 8 · Deploy into AgriOrbit

1. Unzip the download. Copy **`rain_model.onnx`, `scaler.json`, `metrics.json`** into
   `backend/ml/artifacts/` in the repo.
2. Restart the backend — `/api/modelinfo` now reports `available: true` with your metrics.
3. The Advisory page shows the neural forecast next to the official forecast, and
   `/api/mlforecast?lat=…&lon=…` serves predictions for any point in India (or the world).

**Expected reference behaviour:** val accuracy ≈ 0.78–0.86 (day-1) drifting down over the
horizon; Brier clearly below the climatology baseline in monsoon months. Keep the honest
framing in the UI: the model is a *statistical second opinion*, the numerical forecast from
ECMWF/GFS remains the primary source.